# Stage 3 -- Model-Ready: Monthly Means

## Input
`Data/Data_Collection/Final/Stage_2/agg_market_monthly_means.parquet` -- aggregated monthly means table (cap-weighted mean per stock factor + macro monthly factors), keyed on `date`

## Purpose
Applies expanding-window z-standardisation to the aggregated monthly means table to produce the final model-ready monthly means dataset. The approach is analogous to the daily means notebook but uses a 12-month minimum window. Unlike the daily table, the monthly table contains no binary or calendar features, so only the meta columns (`date`, `target_monthly_return`) are skipped.

---

## Pipeline

### Step 1: Load
The Stage 2 aggregated monthly means table is loaded and sorted by date. Shape and date range are reported.

### Step 2: Identify Columns to Z-Score vs Skip
Only `date` and `target_monthly_return` are skipped. The monthly aggregated table contains no binary or calendar features -- those originate from Panel C (macro daily) and do not appear in Panel B (stock monthly) or Panel D (macro monthly). Every other column is z-scored.

**Pre-z-score drop:** `ExchSwitch` is dropped before z-scoring. This is a rare corporate event indicator that is near-constant zero for years, causing the expanding standard deviation to remain near zero and producing undefined z-scores until approximately 2013. It is removed rather than left as a sparse NaN column.

### Step 3: Expanding-Window Z-Standardisation
Applied to all `zscore_cols` using the formula:

`z_t = (x_t - μ_{1:t-1}) / σ_{1:t-1}`

- **`shift(1)` applied to both expanding mean and std** -- the current month is excluded from its own standardisation, preventing look-ahead bias
- **Minimum 12 months (~1 year)** before the first valid z-score is produced
- Computed in one vectorised pass using pandas `expanding().mean()` and `expanding().std()` followed by `shift(1)`
- Expanding arrays deleted immediately after use to free memory
- Any resulting ±inf values (from σ = 0 periods) are replaced with NaN

### Step 4: Drop Warmup Rows
The first `MIN_WINDOW + 1` rows (13 rows) are dropped. If any z-scored columns still contain NaN after this standard trim, an extended diagnostic is run: each offending column is reported with its last NaN row index and date, and rows are trimmed until all NaN are eliminated.

### Step 5: Validate
- **NaN check:** zero NaN expected in all feature columns after warmup trim; any remaining are listed
- **Infinite value check:** confirms no ±inf remain
- **Zero-variance check:** identifies any columns that are all-NaN or constant after z-scoring
- **Target integrity:** mean (~0.008--0.010), std (~0.04--0.05), min, max, annualised Sharpe, NaN count -- confirms target was not z-scored
- **Z-score distribution check:** first 10 z-scored features shown with mean, std, min, max (expect mean ≈ 0, std ≈ 1)
- **No duplicate dates**

### Step 6: Save
Sorted by date and saved to parquet.

---

## Key Design Decisions
- **`MIN_WINDOW = 12` months** (vs 252 days for daily tables), reflecting the monthly cadence. One year of history required before the first z-score is computed.
- **No binary/calendar skip list** because the monthly aggregated table contains none of these features. The skip set contains only the two meta columns.
- **`ExchSwitch` dropped** before z-scoring for the same reason `dlyreti_spread` was dropped in the daily full moments notebook -- near-constant zero produces undefined expanding z-scores for an extended period. This is the only explicit pre-z-score drop in this notebook.
- **Extended NaN diagnostic in warmup trim** identifies exactly which columns are responsible for any NaN persisting beyond the standard warmup, printing each with its last NaN row index and date before trimming.
- **No start-date alignment step** (unlike the daily means notebook which aligns to 2007-11-30). The monthly table's date range is set by its own warmup trim and the combined monthly table does not need to align to a daily spine.

## Output
`Data/Data_Collection/Final/Stage_3_Model_Ready/model_market_monthly_means.parquet` -- keyed on `date` (calendar month-end), all features expanding-window z-standardised using only past data, `target_monthly_return` in raw returns

In [2]:
# %% [markdown]
# # Stage 3 — Model-Ready: Monthly Means
#
# Applies expanding-window z-standardisation to the aggregated monthly means
# table, producing the model-ready monthly dataset.
#
# Z-scoring approach:
#   z_t = (x_t - μ_{1:t-1}) / σ_{1:t-1}
#   - Uses ONLY data up to t-1 (shift(1) ensures no look-ahead)
#   - Minimum 12 months (~1 year) before first valid z-score
#   - Target variable is NOT z-scored (stays in raw returns)
#   - No binary/calendar features in monthly tables (those are in daily Panel C)
#
# Input:  Stage_2/agg_market_monthly_means.parquet
# Output: Stage_3_Model_Ready/model_market_monthly_means.parquet

# %%
import pandas as pd
import numpy as np
from pathlib import Path
import time

IN_PATH = Path('../../../../Data/Data_Collection/Final/Stage_2/agg_market_monthly_means.parquet')
OUT_DIR = Path('../../../../Data/Data_Collection/Final/Stage_3_Model_Ready')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 1: LOAD
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("=" * 90)
print("STEP 1: LOAD")
print("=" * 90)

df = pd.read_parquet(IN_PATH)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

print(f"\n  Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Date range: {df['date'].min().date()} → {df['date'].max().date()}")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 2: IDENTIFY COLUMNS TO Z-SCORE vs SKIP
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 2: IDENTIFY COLUMNS TO Z-SCORE vs SKIP")
print("=" * 90)

# Monthly tables have NO binary/calendar features — those are only in Panel C (daily).
# Skip only meta columns.
meta_cols = ['date', 'target_monthly_return']

all_skip = {c for c in meta_cols if c in df.columns}
zscore_cols = [c for c in df.columns if c not in all_skip]

print(f"\n  Total columns: {df.shape[1]}")
print(f"  Columns to z-score: {len(zscore_cols)}")
print(f"  Columns to skip: {len(all_skip)} ({sorted(all_skip)})")


# Drop ExchSwitch — near-constant zero (rare corporate event),
# expanding σ ≈ 0 for years, making z-scores undefined until 2013
if 'ExchSwitch' in df.columns:
    df = df.drop(columns=['ExchSwitch'])
    zscore_cols = [c for c in zscore_cols if c != 'ExchSwitch']
    print(f"\n  Dropped ExchSwitch (near-zero for years, undefined z-score)")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 3: EXPANDING-WINDOW Z-STANDARDISATION
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 3: EXPANDING-WINDOW Z-STANDARDISATION")
print("=" * 90)

MIN_WINDOW = 12  # 12 months = 1 year

t0 = time.time()

print(f"\n  Z-scoring {len(zscore_cols)} columns with expanding window (min {MIN_WINDOW} months)...")
print(f"  Formula: z_t = (x_t - μ_{{1:t-1}}) / σ_{{1:t-1}}")
print(f"  shift(1) ensures NO look-ahead — current month excluded from mean/std\n")

# Expanding mean and std, shifted by 1 to exclude current observation
expanding_mean = df[zscore_cols].expanding(min_periods=MIN_WINDOW).mean().shift(1)
expanding_std = df[zscore_cols].expanding(min_periods=MIN_WINDOW).std().shift(1)

# Apply z-score
df[zscore_cols] = (df[zscore_cols] - expanding_mean) / expanding_std

del expanding_mean, expanding_std

elapsed = time.time() - t0
print(f"  Z-scoring completed in {elapsed:.1f}s")

# Replace any inf/-inf from division by zero (σ = 0 for constant factors)
inf_before = np.isinf(df[zscore_cols]).sum().sum()
if inf_before > 0:
    df[zscore_cols] = df[zscore_cols].replace([np.inf, -np.inf], np.nan)
    print(f"  ⚠ Replaced {inf_before} infinite values with NaN (from σ = 0 periods)")
else:
    print(f"  ✓ No infinite values produced")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 4: DROP WARMUP ROWS
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 4: DROP WARMUP ROWS")
print("=" * 90)

# First MIN_WINDOW + 1 rows have NaN z-scores (expanding warmup + shift)
warmup_needed = MIN_WINDOW + 1

pre_drop = len(df)
warmup_date = df.iloc[warmup_needed - 1]['date']
df = df.iloc[warmup_needed:].reset_index(drop=True)

print(f"\n  Dropped first {warmup_needed} rows (warmup period)")
print(f"  Warmup end date: {warmup_date.date()}")

# Trim any additional NaN from near-constant factors
nan_rows = df[df[zscore_cols].isna().any(axis=1)]
if len(nan_rows) > 0:
    last_nan_row = max(nan_rows.index)
    pre = len(df)
    
    # Diagnose which columns are causing extended NaN
    print(f"\n  Diagnosing extended NaN (beyond warmup):")
    for c in zscore_cols:
        last_nan = df[df[c].isna()].index
        if len(last_nan) > 0:
            max_row = int(last_nan.max())
            nan_date = df.iloc[max_row]['date'].date()
            total_nan = df[c].isna().sum()
            print(f"    {c:<50s} last NaN row {max_row:>4d} ({nan_date})  total: {total_nan}")
    
    df = df.iloc[last_nan_row + 1:].reset_index(drop=True)
    print(f"\n  Trimmed {pre - len(df)} additional rows to remove early z-score NaN")
else:
    print(f"  ✓ No additional NaN rows to trim")

print(f"\n  Rows: {pre_drop} → {len(df)}")
print(f"  Date range: {df['date'].min().date()} → {df['date'].max().date()}")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 5: VALIDATE
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 5: VALIDATE")
print("=" * 90)

# 5a. NaN check
feature_cols = [c for c in df.columns if c not in ['date', 'target_monthly_return']]
feature_nan = df[feature_cols].isna().sum()
feature_nan_total = feature_nan.sum()

if feature_nan_total > 0:
    nan_cols = feature_nan[feature_nan > 0].sort_values(ascending=False)
    print(f"\n  ⚠ Feature NaN: {feature_nan_total}")
    print(f"  Columns with NaN ({len(nan_cols)}):")
    for c in nan_cols.head(20).index:
        print(f"    {c}: {int(nan_cols[c])}")
else:
    print(f"\n  ✓ Zero NaN in features")

# 5b. Infinite values
inf_count = 0
inf_cols = []
for c in zscore_cols:
    if c in df.columns:
        n_inf = np.isinf(df[c]).sum()
        if n_inf > 0:
            inf_count += n_inf
            inf_cols.append((c, n_inf))

if inf_count > 0:
    print(f"\n  ⚠ Infinite values: {inf_count}")
    for c, n in inf_cols[:15]:
        print(f"    {c}: {n}")
else:
    print(f"  ✓ Zero infinite values")

# 5c. Zero-variance columns
zero_var_cols = []
for c in zscore_cols:
    if c in df.columns:
        if df[c].isna().all():
            zero_var_cols.append(c)
        elif pd.notna(df[c].std()) and float(df[c].std()) == 0:
            zero_var_cols.append(c)

if zero_var_cols:
    print(f"\n  ⚠ Zero-variance after z-score ({len(zero_var_cols)}):")
    for c in zero_var_cols:
        print(f"    {c}")
else:
    print(f"  ✓ No zero-variance columns")

# 5d. Target untouched
print(f"\n  Target statistics (should be raw returns, NOT z-scored):")
print(f"    Mean:   {df['target_monthly_return'].mean():.6f} (expect ~0.008-0.010)")
print(f"    Std:    {df['target_monthly_return'].std():.6f} (expect ~0.04-0.05)")
print(f"    Min:    {df['target_monthly_return'].min():.6f}")
print(f"    Max:    {df['target_monthly_return'].max():.6f}")
print(f"    Sharpe: {df['target_monthly_return'].mean() / df['target_monthly_return'].std() * np.sqrt(12):.2f} (annualised)")
print(f"    NaN:    {df['target_monthly_return'].isna().sum()}")

# 5e. Z-score distribution check
sample_cols = [c for c in zscore_cols if c in df.columns][:10]
print(f"\n  Z-score distribution check (first 10 z-scored features):")
print(f"  {'Column':<40s} {'Mean':>8s} {'Std':>8s} {'Min':>8s} {'Max':>8s}")
print("  " + "-" * 70)
for c in sample_cols:
    vals = df[c].dropna()
    if len(vals) > 0:
        print(f"  {c:<40s} {vals.mean():>8.3f} {vals.std():>8.3f} "
              f"{vals.min():>8.2f} {vals.max():>8.2f}")

# 5f. No duplicate dates
n_dupes = df['date'].duplicated().sum()
assert n_dupes == 0, "FATAL: Duplicate dates!"
print(f"\n  ✓ No duplicate dates")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 6: SAVE
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 6: SAVE")
print("=" * 90)

df = df.sort_values('date').reset_index(drop=True)

out_path = OUT_DIR / 'model_market_monthly_means.parquet'
df.to_parquet(out_path, index=False, engine='pyarrow')

file_size = out_path.stat().st_size
print(f"\n  ✓ Saved: {out_path}")
print(f"    {df.shape[0]} rows × {df.shape[1]} columns")
print(f"    Size: {file_size / 1e3:.1f} KB")

# ═══════════════════════════════════════════════════════════════════════════════
# FINAL SUMMARY
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("MODEL-READY MONTHLY MEANS COMPLETE")
print("=" * 90)

n_zscored = len([c for c in zscore_cols if c in df.columns])

print(f"""
  Input:  agg_market_monthly_means.parquet (Stage 2)
  Output: model_market_monthly_means.parquet (Stage 3)

  Z-standardisation:
    Method:     Expanding window, shift(1), min {MIN_WINDOW} months
    Z-scored:   {n_zscored} features
    Skipped:    {len(all_skip)} (date + target)
    Warmup:     {warmup_needed}+ rows dropped

  Result:
    Rows:       {df.shape[0]} months
    Columns:    {df.shape[1]}
    Dates:      {df['date'].min().date()} → {df['date'].max().date()}
    NaN:        {feature_nan_total} features + {df['target_monthly_return'].isna().sum()} target

  Saved: {out_path}
""")

STEP 1: LOAD

  Loaded: 218 rows × 327 columns
  Date range: 2006-10-31 → 2024-11-30

STEP 2: IDENTIFY COLUMNS TO Z-SCORE vs SKIP

  Total columns: 327
  Columns to z-score: 325
  Columns to skip: 2 (['date', 'target_monthly_return'])

  Dropped ExchSwitch (near-zero for years, undefined z-score)

STEP 3: EXPANDING-WINDOW Z-STANDARDISATION

  Z-scoring 324 columns with expanding window (min 12 months)...
  Formula: z_t = (x_t - μ_{1:t-1}) / σ_{1:t-1}
  shift(1) ensures NO look-ahead — current month excluded from mean/std

  Z-scoring completed in 0.1s
  ✓ No infinite values produced

STEP 4: DROP WARMUP ROWS

  Dropped first 13 rows (warmup period)
  Warmup end date: 2007-10-31
  ✓ No additional NaN rows to trim

  Rows: 218 → 205
  Date range: 2007-11-30 → 2024-11-30

STEP 5: VALIDATE

  ✓ Zero NaN in features
  ✓ Zero infinite values
  ✓ No zero-variance columns

  Target statistics (should be raw returns, NOT z-scored):
    Mean:   0.009844 (expect ~0.008-0.010)
    Std:    0.045218